In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import numpy as np
import kagglehub
#from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
#from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

x_train, y_train = None, None
x_test, y_test = None, None
train_data, test_data = None, None
model = None

def load_modify_data():
    global x_train, y_train
    global x_test, y_test

    gender_mapping = {"Male" : 0, "Female" : 1, "Other" : 2}
    stress_level_mapping = {"Low" : 0, "Medium" : 1, "High" : 2}
    yesno_mapping = {"No" : 0, "Yes" : 1}

    # training data
    train_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/train.csv")
    
    train_data["gender"] = train_data["gender"].map(gender_mapping)
    train_data["stress_level"] = train_data["stress_level"].map(stress_level_mapping)
    train_data["academic_work_impact"] = train_data["academic_work_impact"].map(yesno_mapping)
    
    for field in train_data.drop(columns=["id", "addicted_label"]):
        print(f"Column    >{field}<    in train dataset")
        train_data[field] = train_data[field].fillna(train_data[field].mean())

    x_train = train_data.drop(columns=["id", "addicted_label"])
    y_train = train_data["addicted_label"]
    x_train = np.array(x_train)

    print("---------------------")
    # test data
    test_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/test.csv")

    test_data["gender"] = test_data["gender"].map(gender_mapping)
    test_data["stress_level"] = test_data["stress_level"].map(stress_level_mapping)
    test_data["academic_work_impact"] = test_data["academic_work_impact"].map(yesno_mapping)

    for field in test_data.drop(columns=["id"]):
        print(f"Column    >{field}<    in test dataset")
        test_data[field] = test_data[field].fillna(test_data[field].mean())
    
    x_test = test_data.drop(columns=["id"])
    x_test = np.array(x_test)
    
    print(x_train.shape, x_test.shape)
    
    
load_modify_data()

x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train,
    test_size = 0.2,
    random_state = 42,
    stratify = y_train
)

# Logistic Regression, I got a score of around 0.909
#model = Pipeline([
 #   ("scaler", StandardScaler()),
 #   ("clf", LogisticRegression(max_iter=1000))
#])

#model.fit(x_tr, y_tr)
#pred = model.predict_proba(x_val)[:, 1]
#from sklearn.metrics import roc_auc_score

#score = roc_auc_score(y_val, pred)
#print(score)

# from here down, its just a simple MLP

scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

x_tr = torch.tensor(x_tr, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)

y_tr = torch.tensor(y_tr.to_numpy(), dtype=torch.float32).unsqueeze(1)
y_val = torch.tensor(y_val.to_numpy(), dtype=torch.float32).unsqueeze(1)

device = torch.device("cuda")

x_tr = x_tr.to(device)
x_val = x_val.to(device)
x_test = x_test.to(device)

y_tr = y_tr.to(device)
y_val = y_val.to(device)

# TensorDataset + DataLoader
train_ds = TensorDataset(x_tr, y_tr)
val_ds = TensorDataset(x_val, y_val)

train_loader = DataLoader(train_ds, batch_size = 512, shuffle = True)

val_loader = DataLoader(val_ds, batch_size = 1024, shuffle = False)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(12, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.175),
                                    nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.175),
                                    nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.15),
                                    nn.Linear(128, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.125),
                                    nn.Linear(128, 1))

    def forward(self, x):
        return self.layers(x)

model = MLP().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3 * 2, weight_decay = 0.002)
epochs = 50

for epoch in range(epochs):
    if epoch == 25:
        optimizer.param_groups[0]["lr"] = 1e-3 * 1.5
        
    if epoch == 30:
        optimizer.param_groups[0]["lr"] = 1e-4 * 5.1

    if epoch == 40:
        optimizer.param_groups[0]["lr"] = 1e-5 * 7.5

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    
    model.eval()
    val_loss = 0.0
    preds = []
    labels = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()
            probs = torch.sigmoid(logits)

            preds.append(probs.cpu())
            labels.append(yb.cpu())

    val_loss /= len(val_loader)

    preds = torch.cat(preds).numpy().ravel()
    labels = torch.cat(labels).numpy().ravel()

    auc = roc_auc_score(labels, preds)

    print(f"Epoch {epoch+1:2d} | " f"Train Loss {train_loss:.4f} | " f"Val Loss {val_loss:.4f} | "f"Score {auc:.4f}")


model.eval()

with torch.no_grad():
    logits = model(x_test.to(device))
    probs = torch.sigmoid(logits).cpu().numpy().ravel()

submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv")
submission["addicted_label"] = probs
submission.to_csv("submission.csv", index=False)

Column    >age<    in train dataset
Column    >daily_screen_time_hours<    in train dataset
Column    >social_media_hours<    in train dataset
Column    >gaming_hours<    in train dataset
Column    >work_study_hours<    in train dataset
Column    >sleep_hours<    in train dataset
Column    >notifications_per_day<    in train dataset
Column    >app_opens_per_day<    in train dataset
Column    >weekend_screen_time<    in train dataset
Column    >gender<    in train dataset
Column    >stress_level<    in train dataset
Column    >academic_work_impact<    in train dataset
---------------------
Column    >age<    in test dataset
Column    >daily_screen_time_hours<    in test dataset
Column    >social_media_hours<    in test dataset
Column    >gaming_hours<    in test dataset
Column    >work_study_hours<    in test dataset
Column    >sleep_hours<    in test dataset
Column    >notifications_per_day<    in test dataset
Column    >app_opens_per_day<    in test dataset
Column    >weekend_screen_t